In [ ]:
import numpy as np
import pandas as pd


def generate_ma_crossover_signals(
    data: pd.DataFrame, fast_window: int, slow_window: int
) -> pd.DataFrame:
    """Generates trading signals based on a Moving Average Crossover strategy.

    Parameters:
    - data: pd.DataFrame containing a 'Close' price column.
    - fast_window: The period for the short-term moving average (e.g., 50).
    - slow_window: The period for the long-term moving average (e.g., 200).

    Returns:
    - pd.DataFrame with added MA and Signal columns.
    """
    # Copy data to avoid modifying the original dataframe
    df = data.copy()

    # Calculate the Moving Averages
    df["Fast_MA"] = df["Close"].rolling(window=fast_window).mean()
    df["Slow_MA"] = df["Close"].rolling(window=slow_window).mean()

    # Create a position flag: 1 when Fast is above Slow, 0 otherwise
    df["Position"] = np.where(df["Fast_MA"] > df["Slow_MA"], 1, 0)

    # The signal is the day-to-day change in position
    # +1 indicates Fast crossed ABOVE Slow (BUY)
    # -1 indicates Fast crossed BELOW Slow (SELL)
    df["Signal"] = df["Position"].diff()

    # Clean up the NaN rows caused by the diff() and fill with 0 (Hold)
    df["Signal"] = df["Signal"].fillna(0).astype(int)

    # Optional: Clean up the intermediate 'Position' column if you only want the triggers
    df.drop(columns=["Position"], inplace=True)

    return df